# Exercise 1 (Association Rules) 

In [70]:
def calculate_support(items, transactions): 
    """returns Number_of_occurences/all_transactions"""
    occurence = 0
    for tra in transactions: 
        all_ocuring = True 
        for item in items: 
            if item not in tra: 
                all_ocuring=False 
                break 
        if all_ocuring: 
            occurence +=1 
    
    return occurence/len(transactions)


def get_all_items(transactions): 
    """returns all items from the transactions"""
    itemlist = []
    for t in transactions: 
        for e in t: 
            if e not in itemlist: 
                itemlist.append(e)
    return itemlist

In [71]:
def calculate_L1(transactions, minsup): 
    """returns all itemsets with len=1 and support > minsup"""
    items = get_all_items(transactions)

    L1 = []
    for item in items: 
        supp = calculate_support([item], transactions)
        if minsup <= supp: 
            L1.append([[item],supp])

    return sorted(L1, key=lambda x: x[0])


In [72]:
def apriori_gen(L_k_1): 
    """
    Generiert k-elementige Kandidaten aus (k-1)-elementigen häufigen Mengen
    
    Schritt 1: Verschmelze zwei (k-1)-Mengen wenn sie (k-2) Items gemeinsam haben
    Schritt 2: Entferne Kandidaten, deren (k-1)-Teilmengen nicht häufig sind
    """

    C_k = [] 
    L_list = []
    for item in L_k_1: 
        L_list.append(item[0])

    for i in range(0,len(L_list)): 
        for j in range(i+1, len(L_list)): 

            set1 = sorted(L_list[i])
            set2 = sorted(L_list[j])
            if set1[:-1] == set2[:-1]: 
                candidate = sorted (set(set1) | set(set2) )
                C_k.append(candidate)

    C_k = [list(x) for x in set(tuple(sorted(c)) for c in C_k)] # remove duplikates 
    return C_k 
    

In [73]:
def calculate_confidence(left, right, transactions): 
    combined_supp = calculate_support(left+right, transactions)
    left_supp = calculate_support(left, transactions)

    if left_supp == 0:
        return 0
    
    confidence = combined_supp/ left_supp
    return confidence

In [74]:
##########################################
# I -> all possibles items 
# D -> transactions 
# L_i 
# C_i 

def apriori(transactions, minsupp): 
    # Calculate items 
    items = get_all_items(transactions)
    print(f"Found items: {items}")
    print(f"Minsupp: {minsupp} ({minsupp*100}%)")


    L1 = calculate_L1(transactions, minsupp)
    print("L1 set")
    print(L1)

    L_all = L1.copy()
    L_k_1 = L1.copy() 
    k = 2
    print("\n")
    while L_k_1: 
        print("##########################\n======= Iteration: ", k)
        C_k = apriori_gen(L_k_1)

        if not C_k: 
            print("Didnt found any new C_k")
            break 
        print("C_", k, " : ", C_k)


        L_k = []

        for elem in C_k: 
            supp = calculate_support(elem, transactions)
            if supp >= minsupp: 
                L_k.append([elem,supp])

        if not L_k: 
            print("Didnt found any L_k")
            break 

        print("L_", k, " set")
        print(L_k)
        L_k_1 = L_k 
        L_all.extend(L_k)
        k+=1 
        print("\n")
    print("\n")
    print("Combined sets: ")
    print(L_all)
    return L_all 


In [75]:
def generate_rules(L_all, transactions, minconf): 

    rules = []



    for items, supp in L_all: 
        if len(items)< 2 :  
            continue 

        print("Calc rules from set: ", items)

        for i, resulting_item in enumerate(items): 
            left = items[:i] + items[i+1:] 
            confidence = calculate_confidence(left, [resulting_item], transactions)

            rules.append({
                'left': left,
                'consequence': resulting_item,
                'support': supp,
                'confidence': confidence,
                'valid': confidence >= minconf
            })
    return rules 



def get_recommendations(rules, user_items, all_items, minconf): 

    print("User listened to : ", user_items)

    recommendations = set() 
    for rule in rules: 
        if not rule["valid"]: 
            continue 
        if set(rule['left']) == set(user_items):
            if rule["consequence"] not in user_items: 
                recommendations.add(rule["consequence"])

    if recommendations:
        print(f"\nRecommendations: {sorted(list(recommendations))}")
    else:
        print("No recommendations found")
    
    return recommendations


In [78]:
# Data 
transactions = [
    ['Highway to Hell', 'Country Roads', 'Learn to Fly', 'Dancing Queen'],
    ['Country Roads', 'Learn to Fly'],
    ['Highway to Hell', 'My Heart Will Go On', 'Dancing Queen'],
    ['Dancing Queen', 'Learn to Fly'],
    ['Highway to Hell', 'Learn to Fly', 'Dancing Queen', 'My Heart Will Go On'],
    ['Highway to Hell', 'Learn to Fly']
]

target_transaction = ['Learn to Fly']




print("\n" + "="*60)
print("APRIORI ALGORITHM - MUSIC RECOMMENDATIONS")
print("="*60 + "\n")

minsupp = 0.5
all_itemsets = apriori(transactions, minsupp)

print("\n" + "="*60)
print("Common Sets")
print("="*60)
print(f"\nSets (Support >= {minsupp:.0%}):\n")
for i, (itemset, supp) in enumerate(all_itemsets, 1):
    print(f"  {i}. {itemset}: Support = {supp:.0%}")


minconf = 0.6  
rules = generate_rules(all_itemsets, transactions, minconf)



all_items = get_all_items(transactions)
recommendations = get_recommendations(rules, target_transaction, all_items, minconf)


APRIORI ALGORITHM - MUSIC RECOMMENDATIONS

Found items: ['Highway to Hell', 'Country Roads', 'Learn to Fly', 'Dancing Queen', 'My Heart Will Go On']
Minsupp: 0.5 (50.0%)
L1 set
[[['Dancing Queen'], 0.6666666666666666], [['Highway to Hell'], 0.6666666666666666], [['Learn to Fly'], 0.8333333333333334]]


##########################
======= Iteration:  2
C_ 2  :  [['Dancing Queen', 'Highway to Hell'], ['Highway to Hell', 'Learn to Fly'], ['Dancing Queen', 'Learn to Fly']]
L_ 2  set
[[['Dancing Queen', 'Highway to Hell'], 0.5], [['Highway to Hell', 'Learn to Fly'], 0.5], [['Dancing Queen', 'Learn to Fly'], 0.5]]


##########################
======= Iteration:  3
C_ 3  :  [['Dancing Queen', 'Highway to Hell', 'Learn to Fly']]
Didnt found any L_k


Combined sets: 
[[['Dancing Queen'], 0.6666666666666666], [['Highway to Hell'], 0.6666666666666666], [['Learn to Fly'], 0.8333333333333334], [['Dancing Queen', 'Highway to Hell'], 0.5], [['Highway to Hell', 'Learn to Fly'], 0.5], [['Dancing Queen'